# SegFormer-B0 Scratch Baseline

Notebook huấn luyện SegFormer-B0 từ đầu trên Kvasir-SEG.

- Encoder MiT-B0 khởi tạo ngẫu nhiên
- Binary segmentation với 1 output channel
- Loss: BCE + Dice
- Metrics: Dice và IoU
- Dataset split: 800 / 100 / 100

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'models').exists():
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'Kvasir-SEG'
SAVE_DIR = PROJECT_ROOT / 'experiments' / 'segformer_b0_scratch' / 'checkpoints'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Project:', PROJECT_ROOT)
print('Data:', DATA_DIR)
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from utils.dataset import find_image_mask_pairs, get_dataloaders

image_paths, mask_paths = find_image_mask_pairs(str(DATA_DIR))
assert len(image_paths) == len(mask_paths) and len(image_paths) > 0
print(f'Image-mask pairs: {len(image_paths)}')

train_loader, val_loader, test_loader = get_dataloaders(
    data_dir=str(DATA_DIR),
    img_size=(256, 256),
    batch_size=8,
    seed=42,
)

images, masks = next(iter(train_loader))
print('Images:', tuple(images.shape))
print('Masks :', tuple(masks.shape))

In [ ]:
from models.segformer import build_segformer

model = build_segformer(num_labels=1).to(DEVICE)
with torch.no_grad():
    sample_logits = model(images.to(DEVICE))

print('Logits:', tuple(sample_logits.shape))
print('Trainable parameters:', f'{model.count_parameters():,}')
assert sample_logits.shape == masks.to(DEVICE).shape
print('Smoke test: OK')

## Huấn luyện

Cell dưới đây chạy script train riêng. Model không tải checkpoint pretrained; toàn bộ SegFormer được khởi tạo ngẫu nhiên.

In [ ]:
cmd = [
    sys.executable,
    str(PROJECT_ROOT / 'scripts' / 'train_segformer.py'),
    '--device', DEVICE,
    '--data_dir', str(DATA_DIR),
    '--epochs', '20',
    '--batch_size', '8',
    '--img_size', '256',
    '--lr', '1e-4',
    '--save_dir', str(SAVE_DIR),
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

In [ ]:
import matplotlib.pyplot as plt

history_path = SAVE_DIR / 'history_segformer_b0_scratch.json'
with history_path.open('r', encoding='utf-8') as file:
    history = json.load(file)

epochs = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'], label='Val')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(epochs, history['train_dice'], label='Train')
axes[1].plot(epochs, history['val_dice'], label='Val')
axes[1].set_title('Dice')
axes[1].legend()

axes[2].plot(epochs, history['train_iou'], label='Train')
axes[2].plot(epochs, history['val_iou'], label='Val')
axes[2].set_title('IoU')
axes[2].legend()

for axis in axes:
    axis.set_xlabel('Epoch')
    axis.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print('Checkpoint:', SAVE_DIR / 'best_segformer_b0_scratch.pth')
print('History   :', history_path)
print('Best val Dice:', max(history['val_dice']))
print('Best val IoU :', max(history['val_iou']))